In [1]:
# Import necessary tools
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MultiLabelBinarizer
import pickle
import ipywidgets as widgets
from IPython.display import display, clear_output
import re
import shap
print("TF version:", tf.__version__)
# print("TF Hub version:", hub.__version__)

# Check for GPU availability
print("GPU", "available (YESSSS!!!!!)" if tf.config.list_physical_devices("GPU") else "not available :(")

TF version: 2.10.0
GPU available (YESSSS!!!!!)


In [2]:
# Read the CSVs
weights_df = pd.read_csv("Symptom-severity.csv")
precaution_df = pd.read_csv("symptom_precaution.csv")
description_df = pd.read_csv("symptom_Description.csv")

In [3]:
final_df = pd.read_csv("final_weighted_dataset.csv")

In [4]:
X = final_df.drop("label", axis = 1)
y = final_df["label"]

In [5]:
X

,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,vomiting,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4917,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4918,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
y

0                              Fungal infection
1                              Fungal infection
2                              Fungal infection
3                              Fungal infection
4                              Fungal infection
                         ...                   
4915    (vertigo) Paroymsal  Positional Vertigo
4916                                       Acne
4917                    Urinary tract infection
4918                                  Psoriasis
4919                                   Impetigo
Name: label, Length: 4920, dtype: object

In [7]:
import networkx as nx
import numpy as np
import pandas as pd

# Create graph
G = nx.Graph()

symptoms = X.columns.tolist()
G.add_nodes_from(symptoms)

# Vectorized co-occurrence
co_matrix = np.dot(X.T, X)

for i in range(len(symptoms)):
    for j in range(i+1, len(symptoms)):
        weight = co_matrix[i, j]
        if weight > 0:
            G.add_edge(symptoms[i], symptoms[j], weight=weight)

In [8]:
degree_centrality = nx.degree_centrality(G)
pagerank = nx.pagerank(G, weight='weight')
clustering = nx.clustering(G, weight='weight')

In [9]:
# Convert dicts to aligned arrays
deg_arr = np.array([degree_centrality[s] for s in symptoms])
pr_arr  = np.array([pagerank[s] for s in symptoms])
clust_arr = np.array([clustering[s] for s in symptoms])

# Broadcast multiplication
X_deg   = X.values * deg_arr
X_pr    = X.values * pr_arr
X_clust = X.values * clust_arr

# Combine all features
X_new = np.hstack([
    X.values,     # original
    X_deg,        # degree weighted
    X_pr,         # pagerank weighted
    X_clust       # clustering weighted
])

In [10]:
y_new = y  # exactly same

In [11]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

clf.fit(X_new, y_new)

RandomForestClassifier(n_estimators=200, random_state=42)

In [12]:
import pickle

with open('graph_disease_predictor.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [13]:
import pickle

# Load the model
with open('graph_disease_predictor.pkl', 'rb') as f:
    clf = pickle.load(f)

In [14]:
def array_to_dataframe(input_array, column_names):
    if len(input_array) != len(column_names):
        raise ValueError("Array length and number of columns must match.")
    return pd.DataFrame([input_array], columns=column_names)

In [15]:
# =========================
# SHAP Explainer
# =========================
explainer = shap.TreeExplainer(clf)

# =========================
# Symptom list
# =========================
symptoms = X.columns.to_list()

# =========================
# Create checkboxes
# =========================
checkboxes = [
    widgets.Checkbox(value=False, description=symptom, indent=False)
    for symptom in symptoms
]

# =========================
# UI Elements
# =========================
checkbox_container = widgets.VBox()

search_box = widgets.Text(
    placeholder='Search symptoms...',
    description='Search:',
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

submit_btn = widgets.Button(
    description="Submit",
    button_style='success'
)

# =========================
# Search filter
# =========================
def filter_checkboxes(change):
    search_term = search_box.value.strip().lower()

    if search_term == "":
        filtered = checkboxes
    else:
        pattern = re.compile(r'\b' + re.escape(search_term))
        filtered = [
            cb for cb in checkboxes
            if pattern.search(cb.description.lower())
        ]

    checkbox_container.children = filtered

filter_checkboxes(None)
search_box.observe(filter_checkboxes, names='value')

# =========================
# Submit logic
# =========================
def on_submit(b):
    selected = [cb.description for cb in checkboxes if cb.value]
    binary_array = np.array([1 if cb.value else 0 for cb in checkboxes])

    with output:
        clear_output()

        if not selected:
            print("Please select at least one symptom.")
            return

        print("\033[1mYou selected:\033[0m")
        for s in selected:
            print(f"- {s}")

        # =========================
        # Prepare input
        # =========================
        weighted_input = (
            binary_array *
            np.array(weights_df["weight"].to_numpy())
        )

        # =========================
        # GRAPH FEATURES
        # =========================
        graph_features = []

        for i, symptom in enumerate(X.columns):
            if binary_array[i] == 1:
                graph_features.extend([
                    degree_centrality[symptom],
                    pagerank[symptom],
                    clustering[symptom]
                ])
            else:
                graph_features.extend([0, 0, 0])

        graph_features = np.array(graph_features)

        # =========================
        # FINAL INPUT
        # =========================
        final_input = np.concatenate([
            weighted_input,
            graph_features
        ])

        input_df = pd.DataFrame([final_input])

        # =========================
        # Prediction
        # =========================
        probs = clf.predict_proba(input_df)[0]
        classes = clf.classes_

        top_indices = np.argsort(probs)[::-1][:3]

        prediction = classes[top_indices[0]]
        confidence = probs[top_indices[0]] * 100

        print("\n\033[1mAccording to the model, you might have:\033[0m")
        print(prediction)
        print(f"Prediction confidence: {confidence:.2f}%")

        # =========================
        # SHAP (COMBINED + NORMALIZED)
        # =========================
        shap_values = explainer.shap_values(input_df)

        if isinstance(shap_values, list):
            class_index = list(classes).index(prediction)
            shap_instance = shap_values[class_index][0]
        else:
            shap_instance = shap_values[0]

        N = len(X.columns)

        combined_explanation = []

        for i, symptom in enumerate(X.columns):
            total_value = (
                np.mean(shap_instance[i]) +
                np.mean(shap_instance[i + N]) +
                np.mean(shap_instance[i + 2*N]) +
                np.mean(shap_instance[i + 3*N])
            )

            if symptom in selected:
                combined_explanation.append((symptom, float(total_value)))

        # sort
        combined_explanation.sort(
            key=lambda x: abs(x[1]),
            reverse=True
        )

        # =========================
        # 🔥 NORMALIZATION (FIX 2)
        # =========================
        total_abs = sum(abs(v) for _, v in combined_explanation)

        if total_abs == 0:
            normalized_explanation = [(s, 0) for s, _ in combined_explanation]
        else:
            normalized_explanation = [
                (s, v / total_abs) for s, v in combined_explanation
            ]

        print("\n\033[1mExplanation (Top contributing symptoms):\033[0m")

        for symptom, value in normalized_explanation[:5]:
            effect = "increased" if value > 0 else "reduced"

            print(
                f"- {symptom.replace('_',' ')} "
                f"({effect} likelihood, "
                f"impact={abs(value)*100:.2f}%, "
                f"centrality={degree_centrality[symptom]:.3f})"
            )

        # DEBUG
        print("\n[Debug]")
        print("Max SHAP:", np.max(np.abs(shap_instance)))

        # =========================
        # Description
        # =========================
        print("\n\033[1mDescription:\033[0m")
        row1 = description_df[
            description_df['Disease'] == prediction
        ].iloc[0]

        for col, val in row1.items():
            if col != "Disease":
                print(val)

        # =========================
        # Precautions
        # =========================
        print("\n\033[1mPrecautions:\033[0m")
        row2 = precaution_df[
            precaution_df['Disease'] == prediction
        ].iloc[0]

        for col, val in row2.items():
            if col != "Disease" and pd.notna(val):
                print(f"- {val}")

        # =========================
        # Alternatives
        # =========================
        print("\n\033[1mOther possible conditions (for reference):\033[0m")
        for idx in top_indices[1:]:
            print(
                f"- {classes[idx]} "
                f"({probs[idx]*100:.2f}%)"
            )

        # =========================
        # Disclaimer
        # =========================
        print("\n\033[1mDisclaimer:\033[0m")
        print(
            "This system provides decision support only "
            "and does not replace professional medical diagnosis. "
            "Please consult a qualified healthcare professional."
        )

# =========================
# Connect button
# =========================
submit_btn.on_click(on_submit)

# =========================
# Display UI
# =========================
display(
    widgets.HTML("<h4>Enter your symptoms</h4>"),
    search_box,
    widgets.Box(
        [checkbox_container],
        layout=widgets.Layout(height='400px', overflow_y='scroll')
    ),
    submit_btn,
    output
)

HTML(value='<h4>Enter your symptoms</h4>')

Text(value='', description='Search:', layout=Layout(width='50%'), placeholder='Search symptoms...')

Box(children=(VBox(children=(Checkbox(value=False, description='abdominal_pain', indent=False), Checkbox(value…

Button(button_style='success', description='Submit', style=ButtonStyle())

Output()